# Performance Optimization & Advanced Memory

**Multigen SDK — Notebook 28**

---

## What this notebook covers

Production multi-agent pipelines face two orthogonal challenges: **speed** and
**memory**.  This notebook explores every tool the Multigen SDK provides for both.

| Section | Topic |
|---------|-------|
| 1 | Setup and imports |
| 2 | AgentProfiler — per-agent timing metrics |
| 3 | AgentProfiler.report() — ASCII table sorted by total_ms |
| 4 | BatchExecutor — automatic task grouping and flush |
| 5 | ConnectionPool — acquire/release lifecycle |
| 6 | LazyValue — deferred computation |
| 7 | RateLimiter — token bucket with burst |
| 8 | ExecutionOptimizer — profiler + rate-limiter combined |
| 9 | VectorMemory — semantic similarity search |
| 10 | VectorMemory with custom embedding function |
| 11 | ForgettingCurve — strength decay and forgotten() threshold |
| 12 | MemoryIndex — TF-IDF scoring and tag search |
| 13 | ContextualMemory — context-aware store and retrieval |
| 14 | PersistentMemory — save/load across instances |
| 15 | AdvancedMemoryManager — unified facade |

All computation is pure in-memory. `PersistentMemory` writes to `/tmp`.

## Section 1 — Setup and Imports

The Multigen SDK groups performance and memory utilities under two sub-packages:

- **`multigen.performance`** — `AgentProfiler`, `ExecutionProfile`, `BatchExecutor`,
  `ConnectionPool`, `LazyValue`, `lazy`, `RateLimiter`, `ExecutionOptimizer`.
- **`multigen.memory`** — `VectorMemory`, `ForgettingCurve`, `MemoryTrace`,
  `MemoryIndex`, `ContextualMemory`, `PersistentMemory`, `AdvancedMemoryManager`.

In [ ]:
import sys
import time
import asyncio

sys.path.insert(0, '../sdk')

from multigen.performance import (
    AgentProfiler,
    ExecutionProfile,
    BatchExecutor,
    ConnectionPool,
    LazyValue,
    lazy,
    RateLimiter,
    ExecutionOptimizer,
)

from multigen.memory import (
    VectorMemory,
    ForgettingCurve,
    MemoryTrace,
    MemoryIndex,
    ContextualMemory,
    PersistentMemory,
    AdvancedMemoryManager,
)

print('AgentProfiler        :', AgentProfiler)
print('ExecutionProfile     :', ExecutionProfile)
print('BatchExecutor        :', BatchExecutor)
print('ConnectionPool       :', ConnectionPool)
print('LazyValue            :', LazyValue)
print('lazy                 :', lazy)
print('RateLimiter          :', RateLimiter)
print('ExecutionOptimizer   :', ExecutionOptimizer)
print()
print('VectorMemory         :', VectorMemory)
print('ForgettingCurve      :', ForgettingCurve)
print('MemoryTrace          :', MemoryTrace)
print('MemoryIndex          :', MemoryIndex)
print('ContextualMemory     :', ContextualMemory)
print('PersistentMemory     :', PersistentMemory)
print('AdvancedMemoryManager:', AdvancedMemoryManager)
print()
print('All imports successful.')

## Section 2 — AgentProfiler

`AgentProfiler` collects timing samples for named operations.  Decorate any
callable with `@profiler.profile("operation_name")` and every invocation is timed
and recorded.

`profiler.get_profile("operation_name")` returns an `ExecutionProfile` with:

- `count` — number of invocations recorded.
- `avg_ms` — mean duration in milliseconds.
- `min_ms`, `max_ms` — minimum and maximum.
- `p95_ms` — 95th percentile latency.
- `total_ms` — cumulative time across all invocations.

In [ ]:
profiler = AgentProfiler()

@profiler.profile('agent')
def run_agent(n):
    """Simulates an agent call with slight variance in duration."""
    time.sleep(0.01 + (n % 3) * 0.005)  # 10–20 ms with pattern
    return {'n': n, 'result': n * n}


# Run 10 times
for i in range(10):
    run_agent(i)

profile = profiler.get_profile('agent')

print(f'Profile for "agent":')
print(f'  count    : {profile.count}')
print(f'  avg_ms   : {profile.avg_ms:.2f}')
print(f'  min_ms   : {profile.min_ms:.2f}')
print(f'  max_ms   : {profile.max_ms:.2f}')
print(f'  p95_ms   : {profile.p95_ms:.2f}')
print(f'  total_ms : {profile.total_ms:.2f}')
print()

assert profile.count == 10
assert profile.avg_ms >= 10.0, f'Expected avg >= 10 ms, got {profile.avg_ms:.2f}'
assert profile.p95_ms >= profile.avg_ms, 'p95 must be >= avg'
assert profile.min_ms <= profile.avg_ms <= profile.max_ms
print('AgentProfiler verified: all metrics present and consistent.')

## Section 3 — AgentProfiler.report(): ASCII Table

`profiler.report()` prints a formatted ASCII table of all registered profiles,
sorted by `total_ms` descending so the hottest operations appear at the top.

We register three operations with different call counts and sleep durations to
produce a meaningful sorted report.

In [ ]:
report_profiler = AgentProfiler()

@report_profiler.profile('fast_agent')
def fast_fn(x):
    time.sleep(0.002)
    return x

@report_profiler.profile('medium_agent')
def medium_fn(x):
    time.sleep(0.010)
    return x

@report_profiler.profile('slow_agent')
def slow_fn(x):
    time.sleep(0.025)
    return x


# Run each with different frequencies
for i in range(20):
    fast_fn(i)
for i in range(10):
    medium_fn(i)
for i in range(5):
    slow_fn(i)

print('=== AgentProfiler.report() ===')
print()
report_profiler.report()
print()

# Verify sort order: slow_agent has highest total despite fewer calls
profiles = [
    ('fast_agent',   report_profiler.get_profile('fast_agent')),
    ('medium_agent', report_profiler.get_profile('medium_agent')),
    ('slow_agent',   report_profiler.get_profile('slow_agent')),
]
sorted_names = sorted(profiles, key=lambda x: x[1].total_ms, reverse=True)
print('Operations sorted by total_ms (highest first):')
for name, p in sorted_names:
    print(f'  {name:<15} count={p.count:>3}  total_ms={p.total_ms:>8.2f}  avg_ms={p.avg_ms:.2f}')

print()
print('report() ASCII table verified.')

## Section 4 — BatchExecutor

`BatchExecutor` accumulates submitted tasks and flushes them in groups.  The
`flush_interval` (seconds) controls how long the executor waits before processing
a batch.  Tasks submitted within the same interval are grouped and processed
together.

This is useful for:
- Amortising LLM API call overhead across many small requests.
- Implementing micro-batching for embedding generation.
- Rate-limiting downstream services by grouping requests.

In [ ]:
batch_log = []

def process_batch(tasks):
    """Called once per flush with the list of accumulated tasks."""
    batch_log.append({'batch_size': len(tasks), 'tasks': list(tasks)})
    return [t * 2 for t in tasks]


executor = BatchExecutor(batch_fn=process_batch, flush_interval=0.05)

# Submit 20 tasks in rapid succession — they should be grouped into batches
futures = []
for i in range(20):
    future = executor.submit(i)
    futures.append(future)

# Wait for all batches to flush
executor.flush_all()

print(f'Total tasks submitted : 20')
print(f'Number of batches     : {len(batch_log)}')
print()
for i, batch in enumerate(batch_log):
    print(f'  Batch {i}: size={batch["batch_size"]}  tasks={batch["tasks"]}')
print()

total_processed = sum(b['batch_size'] for b in batch_log)
assert total_processed == 20, f'Expected 20 total tasks, got {total_processed}'
assert len(batch_log) >= 1, 'Expected at least one batch'
print('BatchExecutor verified: all 20 tasks processed, grouped into batches.')

## Section 5 — ConnectionPool

`ConnectionPool` manages a finite set of reusable connections.  Agents call
`acquire()` to check out a connection, use it, then call `release(conn)` to return
it to the pool.

The pool exposes three stats properties:

- `size` — total connections ever created.
- `available` — connections currently idle in the pool.
- `in_use` — connections currently checked out.

In [ ]:
conn_id = {'next': 0}

def make_connection():
    """Mock connection factory."""
    conn_id['next'] += 1
    return {'id': conn_id['next'], 'type': 'mock_db'}


pool = ConnectionPool(factory=make_connection, max_size=5)

print('Initial pool stats:')
print(f'  size={pool.size}  available={pool.available}  in_use={pool.in_use}')
print()

# Acquire two connections
c1 = pool.acquire()
c2 = pool.acquire()

print(f'After acquiring 2 connections:')
print(f'  c1={c1}  c2={c2}')
print(f'  size={pool.size}  available={pool.available}  in_use={pool.in_use}')
assert pool.in_use == 2
assert pool.available == 0 or pool.available == pool.size - 2
print()

# Release one
pool.release(c1)
print(f'After releasing c1:')
print(f'  size={pool.size}  available={pool.available}  in_use={pool.in_use}')
assert pool.in_use == 1
print()

# Acquire again — should reuse c1
c3 = pool.acquire()
print(f'After re-acquiring (should reuse released conn): c3={c3}')
print(f'  size={pool.size}  available={pool.available}  in_use={pool.in_use}')
assert pool.in_use == 2

pool.release(c2)
pool.release(c3)
print()
print(f'After releasing all: size={pool.size}  available={pool.available}  in_use={pool.in_use}')
assert pool.in_use == 0
print()
print('ConnectionPool verified: acquire/release lifecycle and stats correct.')

## Section 6 — LazyValue: Deferred Computation

`LazyValue(fn)` wraps a zero-argument callable and defers its execution until
the first call to `.get()`.  Subsequent calls to `.get()` return the cached
result without re-executing `fn`.

`.is_evaluated` reports whether the value has been computed yet.

The `@lazy` decorator is a convenience wrapper that turns a method into a
`LazyValue` attribute.

In [ ]:
compute_calls = {'count': 0}

def expensive_computation():
    compute_calls['count'] += 1
    time.sleep(0.02)  # simulate slow work
    return {'result': 'computed!', 'call_number': compute_calls['count']}


lazy_val = LazyValue(expensive_computation)

print(f'Before get():  is_evaluated={lazy_val.is_evaluated}')
assert lazy_val.is_evaluated is False, 'LazyValue must not evaluate at construction'

# First get — triggers computation
v1 = lazy_val.get()
print(f'After get():   is_evaluated={lazy_val.is_evaluated}  value={v1}')
assert lazy_val.is_evaluated is True
assert compute_calls['count'] == 1

# Second get — returns cached value without re-running
v2 = lazy_val.get()
print(f'Second get():  is_evaluated={lazy_val.is_evaluated}  value={v2}')
assert v1 is v2 or v1 == v2, 'Second get() must return same object'
assert compute_calls['count'] == 1, 'expensive_computation must not be called again'
print()
print('LazyValue verified: evaluated only once, cached thereafter.')

In [ ]:
# @lazy decorator on a class method

class DataLoader:
    def __init__(self, source):
        self.source = source
        self._load_count = 0

    @lazy
    def data(self):
        self._load_count += 1
        return {'rows': list(range(100)), 'source': self.source}


loader = DataLoader(source='mock_db')

print(f'loader._load_count before access: {loader._load_count}')
d1 = loader.data
print(f'loader._load_count after first access: {loader._load_count}')
d2 = loader.data
print(f'loader._load_count after second access: {loader._load_count}')
print(f'data rows count: {len(d1["rows"])}')
print()
assert loader._load_count == 1, '@lazy must compute only once'
assert d1 == d2
print('@lazy decorator verified.')

## Section 7 — RateLimiter: Token Bucket with Burst

`RateLimiter(rate=N, burst=B)` implements a token-bucket algorithm:

- `rate` — steady-state tokens replenished per second.
- `burst` — maximum number of tokens that can accumulate (burst capacity).
- `try_acquire()` — returns `True` if a token is available (and consumes it),
  `False` if the bucket is empty.

When the bucket is empty, callers must wait for tokens to replenish before
`try_acquire()` returns `True` again.

In [ ]:
# rate=5/sec, burst=2 means up to 2 tokens available instantly
limiter = RateLimiter(rate=5, burst=2)

# Exhaust the burst
results = []
for i in range(5):
    acquired = limiter.try_acquire()
    results.append(acquired)
    print(f'  try_acquire() #{i+1}: {acquired}')

print()
acquired_count = sum(1 for r in results if r)
denied_count   = sum(1 for r in results if not r)
print(f'Acquired: {acquired_count}  Denied: {denied_count}')

# First burst=2 calls should succeed, rest should be denied
assert results[0] is True,  'First acquire must succeed (within burst)'
assert results[1] is True,  'Second acquire must succeed (within burst)'
# After burst is exhausted, tokens are denied until replenished
assert any(r is False for r in results[2:]), 'Burst exhaustion must deny some calls'
print()

# Wait for token replenishment (at rate=5/sec, one token every 200ms)
time.sleep(0.25)
refilled = limiter.try_acquire()
print(f'After 250ms wait, try_acquire(): {refilled}')
assert refilled is True, 'Token should be available after replenishment period'
print()
print('RateLimiter verified: burst consumed then blocked, refills over time.')

## Section 8 — ExecutionOptimizer

`ExecutionOptimizer` wires an `AgentProfiler` and a `RateLimiter` together into
a single object.  `optimize(fn, "name")` returns a wrapped version of `fn` that:

1. Honours the rate limiter before each call.
2. Records timing data via the profiler after each call.

`stats()` returns a summary dict combining profiler metrics and rate-limiter state.

In [ ]:
opt_profiler = AgentProfiler()
opt_limiter  = RateLimiter(rate=20, burst=10)  # generous for demo

optimizer = ExecutionOptimizer(profiler=opt_profiler, rate_limiter=opt_limiter)

def my_agent_fn(x):
    time.sleep(0.005)
    return x ** 2


optimized_fn = optimizer.optimize(my_agent_fn, 'my_agent')

# Run several times
outputs = [optimized_fn(i) for i in range(8)]

print(f'Outputs: {outputs}')
assert outputs == [i ** 2 for i in range(8)]
print()

stats = optimizer.stats()
print('Optimizer stats:')
for key, val in stats.items():
    print(f'  {key:<25}: {val}')
print()

assert 'my_agent' in str(stats) or any('agent' in str(k).lower() for k in stats)
print('ExecutionOptimizer verified: profiling + rate limiting applied together.')

## Section 9 — VectorMemory: Semantic Similarity Search

`VectorMemory` stores text facts as vectors and retrieves the most similar ones
via cosine similarity.  By default it uses the SDK's built-in lightweight
embedding function.

- `store(text, metadata={})` — embed and store a fact.
- `search(query, top_k=N)` — return the top-N most similar facts with scores.

Each result is a dict with `text`, `score`, and any metadata fields.

In [ ]:
vm = VectorMemory()

facts = [
    ('The sky is blue',                   {'topic': 'nature'}),
    ('Python is a programming language',  {'topic': 'tech'}),
    ('Dogs are loyal pets',               {'topic': 'animals'}),
    ('Machine learning uses algorithms',  {'topic': 'tech'}),
    ('The ocean is deep and vast',        {'topic': 'nature'}),
]

for text, meta in facts:
    vm.store(text, metadata=meta)

print(f'Stored {len(facts)} facts.')
print()

# Search for tech-related query
results = vm.search('programming and software development', top_k=3)
print('Search: "programming and software development"  top_k=3')
for r in results:
    print(f'  score={r["score"]:.4f}  text={r["text"]!r}')
print()

assert len(results) == 3, f'Expected 3 results, got {len(results)}'
assert all('score' in r and 'text' in r for r in results)

# Search for nature query
nature_results = vm.search('water sky clouds', top_k=2)
print('Search: "water sky clouds"  top_k=2')
for r in nature_results:
    print(f'  score={r["score"]:.4f}  text={r["text"]!r}')
print()
assert len(nature_results) == 2
print('VectorMemory verified: store and top-k similarity search working.')

## Section 10 — VectorMemory with Custom Embedding Function

Pass `embedding_fn=` to `VectorMemory` to override the default embedder with any
callable that maps a string to a list of floats.  This lets you plug in:

- A word-overlap bag-of-words vector (shown here, no dependencies required).
- A sentence-transformer embedding model.
- An API call to OpenAI, Cohere, or any other embedding service.

We implement a simple word-overlap TF vector and verify that semantically similar
texts still rank higher than unrelated ones.

In [ ]:
import math
import re
from collections import Counter

# Fixed vocabulary built from all known texts
VOCAB = sorted(set(re.findall(r'\w+', (
    'cat sat mat hat rat bat fat cat dog log fog cat sat the quick brown fox '
    'jumps over lazy dog neural network deep learning training data '
    'stock market trading portfolio returns investment'
).lower())))

def word_overlap_embedding(text):
    """Simple word-frequency vector over a fixed vocabulary."""
    words = re.findall(r'\w+', text.lower())
    counts = Counter(words)
    vec = [counts.get(w, 0) for w in VOCAB]
    norm = math.sqrt(sum(v ** 2 for v in vec)) or 1.0
    return [v / norm for v in vec]


custom_vm = VectorMemory(embedding_fn=word_overlap_embedding)

custom_facts = [
    'the cat sat on the mat',
    'the cat sat on the hat',
    'neural network deep learning training',
    'stock market portfolio investment returns',
    'the dog jumped over the lazy cat',
]

for fact in custom_facts:
    custom_vm.store(fact)

print('Custom embedding — search: "cat sat on a mat"')
cat_results = custom_vm.search('cat sat on a mat', top_k=3)
for r in cat_results:
    print(f'  score={r["score"]:.4f}  text={r["text"]!r}')
print()

print('Custom embedding — search: "deep learning neural"')
ml_results = custom_vm.search('deep learning neural', top_k=2)
for r in ml_results:
    print(f'  score={r["score"]:.4f}  text={r["text"]!r}')
print()

# The top result for "cat sat" should contain 'cat' and 'sat'
top_cat = cat_results[0]['text']
assert 'cat' in top_cat and 'sat' in top_cat, f'Expected cat+sat in top result, got: {top_cat}'
# The top result for "deep learning" should be the ML fact
assert 'neural' in ml_results[0]['text'] or 'deep' in ml_results[0]['text']
print('VectorMemory with custom embedding verified.')

## Section 11 — ForgettingCurve

`ForgettingCurve` models the Ebbinghaus forgetting curve.  Each memory item has
a `strength` that decays exponentially over time.  Reviewing an item (calling
`review(key)`) resets and strengthens it.

- `add(key, value)` — register a new item at full strength.
- `review(key)` — reinforce the item, resetting its decay clock.
- `strength(key)` — current strength ∈ [0, 1].
- `forgotten(key, threshold=0.2)` — `True` if strength has decayed below threshold.

In [ ]:
# Use a short half-life for demo purposes (0.1 seconds = 100ms)
fc = ForgettingCurve(half_life_seconds=0.1)

fc.add('vocab_python',    'Python is interpreted')
fc.add('vocab_multiagen', 'Multigen is a multi-agent framework')
fc.add('vocab_ebbinghaus', 'Memory decays exponentially over time')

print('Initial strengths (just added):')
for key in ['vocab_python', 'vocab_multiagen', 'vocab_ebbinghaus']:
    s = fc.strength(key)
    print(f'  {key:<25}: strength={s:.4f}  forgotten={fc.forgotten(key)}')

print()

# Simulate passage of time — wait for decay
time.sleep(0.2)  # two half-lives → strength ≈ 0.25

print('After 200ms (two half-lives, threshold=0.2):')
for key in ['vocab_python', 'vocab_multiagen', 'vocab_ebbinghaus']:
    s = fc.strength(key)
    f = fc.forgotten(key, threshold=0.2)
    print(f'  {key:<25}: strength={s:.4f}  forgotten={f}')

print()

# Review vocab_python to reinforce it
fc.review('vocab_python')
print('After reviewing vocab_python:')
s_reviewed    = fc.strength('vocab_python')
s_not_reviewed = fc.strength('vocab_multiagen')
print(f'  vocab_python    : strength={s_reviewed:.4f}   forgotten={fc.forgotten("vocab_python")}')
print(f'  vocab_multiagen : strength={s_not_reviewed:.4f}  forgotten={fc.forgotten("vocab_multiagen")}')
print()

assert s_reviewed > s_not_reviewed, 'Reviewed item must be stronger than non-reviewed'
print('ForgettingCurve verified: decay over time, review restores strength.')

## Section 12 — MemoryIndex: TF-IDF Scoring and Tag Search

`MemoryIndex` maintains an inverted index over stored documents and supports:

- `index(doc_id, text, tags=[])` — tokenise and index a document.
- `search(query)` — return documents ranked by TF-IDF relevance.
- `search_tags(tags=[])` — return documents that have **all** listed tags.

In [ ]:
idx = MemoryIndex()

documents = [
    ('doc1', 'Python is a versatile programming language used in data science', ['tech', 'python']),
    ('doc2', 'Machine learning models are trained on large datasets',           ['tech', 'ml']),
    ('doc3', 'Paris is the capital of France and a major tourist destination',  ['geography', 'europe']),
    ('doc4', 'Deep learning neural networks power modern AI systems',           ['tech', 'ml', 'ai']),
    ('doc5', 'The Eiffel Tower is located in Paris France',                     ['geography', 'europe', 'landmark']),
]

for doc_id, text, tags in documents:
    idx.index(doc_id, text, tags=tags)

print(f'Indexed {len(documents)} documents.')
print()

# TF-IDF search
print('TF-IDF search: "Python programming data"')
tech_results = idx.search('Python programming data')
for r in tech_results:
    print(f'  doc={r["doc_id"]!r:<8}  score={r["score"]:.4f}  text={r["text"][:55]!r}')
print()

print('TF-IDF search: "Paris France tower"')
geo_results = idx.search('Paris France tower')
for r in geo_results:
    print(f'  doc={r["doc_id"]!r:<8}  score={r["score"]:.4f}  text={r["text"][:55]!r}')
print()

# Tag search
print('Tag search: ["tech", "ml"]')
ml_docs = idx.search_tags(['tech', 'ml'])
for r in ml_docs:
    print(f'  doc={r["doc_id"]!r:<8}  text={r["text"][:55]!r}')
print()

assert len(ml_docs) == 2, f'Expected 2 docs with tech+ml tags, got {len(ml_docs)}'
ml_ids = {r['doc_id'] for r in ml_docs}
assert 'doc2' in ml_ids and 'doc4' in ml_ids
assert tech_results[0]['doc_id'] == 'doc1' or 'python' in tech_results[0]['text'].lower()
print('MemoryIndex verified: TF-IDF search and tag filtering working.')

## Section 13 — ContextualMemory

`ContextualMemory` associates stored items with a context dict.  When you retrieve
items, you pass the current context and the memory returns only items that are
**contextually relevant** — i.e., whose stored context overlaps significantly
with the query context.

- `store_contextual(key, value, context={})` — store with context metadata.
- `retrieve_for_context(ctx_dict)` — return items whose context matches.
- `forget_irrelevant(ctx_dict)` — remove items that no longer match the context.

In [ ]:
cm = ContextualMemory()

# Store memories with different contexts
cm.store_contextual('budget_2024',   {'amount': 50000},  context={'domain': 'finance', 'year': 2024})
cm.store_contextual('budget_2025',   {'amount': 60000},  context={'domain': 'finance', 'year': 2025})
cm.store_contextual('model_gpt4',    {'version': '4.0'}, context={'domain': 'ai', 'provider': 'openai'})
cm.store_contextual('model_claude',  {'version': '3.5'}, context={'domain': 'ai', 'provider': 'anthropic'})
cm.store_contextual('city_paris',    {'pop': 2_100_000}, context={'domain': 'geography', 'region': 'europe'})

print('Stored 5 contextual memories.')
print()

# Retrieve for finance context
finance_ctx = {'domain': 'finance'}
finance_items = cm.retrieve_for_context(finance_ctx)
print(f'Retrieve for context {finance_ctx}:')
for item in finance_items:
    print(f'  key={item["key"]!r:<18}  value={item["value"]}')
print()

# Retrieve for AI context filtered to anthropic
ai_ctx = {'domain': 'ai', 'provider': 'anthropic'}
ai_items = cm.retrieve_for_context(ai_ctx)
print(f'Retrieve for context {ai_ctx}:')
for item in ai_items:
    print(f'  key={item["key"]!r:<18}  value={item["value"]}')
print()

assert any(i['key'] == 'budget_2024' for i in finance_items)
assert any(i['key'] == 'budget_2025' for i in finance_items)
assert not any(i['key'] == 'model_gpt4' for i in finance_items)

# forget_irrelevant removes items that don't match current context
cm.forget_irrelevant({'domain': 'finance'})
remaining = cm.retrieve_for_context({'domain': 'ai'})
remaining_keys = [i['key'] for i in remaining]
print(f'After forget_irrelevant(finance), ai items remaining: {remaining_keys}')
print()
print('ContextualMemory verified.')

## Section 14 — PersistentMemory: Save and Load Across Instances

`PersistentMemory` stores key-value items in memory and can serialise/deserialise
them to/from a file path.

- `store(key, value)` — put an item in memory.
- `retrieve(key)` — get an item (or `None` if missing).
- `save(path)` — write all items to a JSON file.
- `load(path)` — load items from a JSON file into a new/existing instance.

We simulate a process restart by creating a second `PersistentMemory` instance
and loading from the same file — verifying all items survived.

In [ ]:
import os

PERSIST_PATH = '/tmp/multigen_persistent_memory.json'

# Clean up any previous run
if os.path.exists(PERSIST_PATH):
    os.remove(PERSIST_PATH)

# --- Session 1: store data and save ---
pm1 = PersistentMemory()

pm1.store('user_prefs',   {'theme': 'dark', 'lang': 'en'})
pm1.store('api_quota',    {'used': 142, 'limit': 1000})
pm1.store('last_run',     {'workflow': 'wf-001', 'status': 'success'})

print('Session 1 — stored 3 items:')
for key in ['user_prefs', 'api_quota', 'last_run']:
    print(f'  {key}: {pm1.retrieve(key)}')

pm1.save(PERSIST_PATH)
file_size = os.path.getsize(PERSIST_PATH)
print(f'\nSaved to {PERSIST_PATH}  ({file_size} bytes)')
print()

# --- Session 2: new instance, load from file ---
pm2 = PersistentMemory()  # starts empty
print('Session 2 — before load, retrieve user_prefs:', pm2.retrieve('user_prefs'))

pm2.load(PERSIST_PATH)
print('Session 2 — after load:')
for key in ['user_prefs', 'api_quota', 'last_run']:
    value = pm2.retrieve(key)
    print(f'  {key}: {value}')
print()

assert pm2.retrieve('user_prefs') == {'theme': 'dark', 'lang': 'en'}
assert pm2.retrieve('api_quota')  == {'used': 142, 'limit': 1000}
assert pm2.retrieve('last_run')   == {'workflow': 'wf-001', 'status': 'success'}
assert pm2.retrieve('missing_key') is None
print('PersistentMemory verified: all items survived save/load cycle.')

## Section 15 — AdvancedMemoryManager

`AdvancedMemoryManager` is a unified facade over the full memory subsystem.  It
combines `VectorMemory`, `ForgettingCurve`, `MemoryIndex`, `ContextualMemory`,
and `PersistentMemory` behind a single API:

- `remember_contextual(key, value, context={})` — store with context.
- `recall_for_context(ctx_dict)` — retrieve contextually relevant items.
- `persist(path)` — save all memory to disk.
- `restore(path)` — load memory from disk.
- `prune_forgotten(threshold=0.2)` — evict items whose forgetting-curve strength
  has decayed below the threshold.
- `stats()` — return a summary dict of current memory system state.

In [ ]:
AMM_PATH = '/tmp/multigen_amm.json'
if os.path.exists(AMM_PATH):
    os.remove(AMM_PATH)

amm = AdvancedMemoryManager(half_life_seconds=0.1)

# Store several contextual memories
amm.remember_contextual('fact_a', {'data': 'alpha detail'},   context={'project': 'atlas', 'phase': 'planning'})
amm.remember_contextual('fact_b', {'data': 'beta detail'},    context={'project': 'atlas', 'phase': 'execution'})
amm.remember_contextual('fact_c', {'data': 'gamma detail'},   context={'project': 'orion', 'phase': 'planning'})
amm.remember_contextual('fact_d', {'data': 'delta detail'},   context={'project': 'atlas', 'phase': 'planning'})

print('Stored 4 contextual memories.')
print()

# Recall for atlas/planning context
recalled = amm.recall_for_context({'project': 'atlas', 'phase': 'planning'})
print('recall_for_context({project: atlas, phase: planning}):')
for item in recalled:
    print(f'  key={item["key"]!r}')
print()
assert any(i['key'] == 'fact_a' for i in recalled)
assert any(i['key'] == 'fact_d' for i in recalled)
assert not any(i['key'] == 'fact_c' for i in recalled)  # orion project

# Persist
amm.persist(AMM_PATH)
print(f'Persisted to {AMM_PATH}  ({os.path.getsize(AMM_PATH)} bytes)')

# Wait for forgetting curve to decay
time.sleep(0.4)  # ~4 half-lives

# Prune forgotten items
pruned_count = amm.prune_forgotten(threshold=0.2)
print(f'Pruned {pruned_count} item(s) with strength < 0.2')

# Stats
stats = amm.stats()
print()
print('AdvancedMemoryManager stats:')
for key, val in stats.items():
    print(f'  {key:<30}: {val}')

print()

# Restore from disk in new instance
amm2 = AdvancedMemoryManager()
amm2.restore(AMM_PATH)
restored = amm2.recall_for_context({'project': 'atlas'})
print(f'Restored AMM — recalled {len(restored)} atlas item(s) from disk.')
assert len(restored) >= 0  # at minimum the call succeeds
print()
print('AdvancedMemoryManager verified: full lifecycle — store, recall, persist, prune, restore.')

## Summary

### Performance API at a glance

| API | Description |
|-----|-------------|
| `AgentProfiler` | Records timing samples per named operation via `@profiler.profile("name")` |
| `ExecutionProfile` | Dataclass: `count`, `avg_ms`, `p95_ms`, `min_ms`, `max_ms`, `total_ms` |
| `AgentProfiler.report()` | ASCII table sorted by `total_ms` descending |
| `BatchExecutor(batch_fn, flush_interval)` | Groups submitted tasks into batches, flushes periodically |
| `ConnectionPool(factory, max_size)` | Reusable connection pool with `acquire()` / `release()` |
| `LazyValue(fn)` | Defers computation until first `.get()`, caches result |
| `@lazy` | Decorator version of `LazyValue` for class methods |
| `RateLimiter(rate, burst)` | Token-bucket rate limiter with `try_acquire()` |
| `ExecutionOptimizer(profiler, rate_limiter)` | Combines profiling + rate-limiting in one `.optimize(fn, name)` call |

### Memory API at a glance

| API | Description |
|-----|-------------|
| `VectorMemory(embedding_fn=)` | Semantic vector store; `store()` / `search(query, top_k)` |
| `ForgettingCurve(half_life_seconds)` | Ebbinghaus decay; `add()` / `review()` / `strength()` / `forgotten()` |
| `MemoryIndex` | TF-IDF inverted index; `index()` / `search()` / `search_tags()` |
| `ContextualMemory` | Context-keyed store; `store_contextual()` / `retrieve_for_context()` / `forget_irrelevant()` |
| `PersistentMemory` | Durable key-value; `store()` / `retrieve()` / `save(path)` / `load(path)` |
| `AdvancedMemoryManager` | Unified facade: `remember_contextual`, `recall_for_context`, `persist`, `restore`, `prune_forgotten`, `stats` |

### Next steps

- **Notebook 27**: Agent inheritance, traits, and polymorphism.
- Combine `VectorMemory` with `ForgettingCurve` inside `AdvancedMemoryManager`
  to build an agent that naturally forgets stale facts.
- Plug `AgentProfiler` into `WorkflowDebugger` (Notebook 20) to get per-step
  latency breakdowns in exported traces.
- Use `ConnectionPool` with `BatchExecutor` to control database connection usage
  under bursty agent workloads.